# CellSight AI — Starter Pipeline

`pip install pandas scikit-learn requests joblib`

# Part A — baseline pipeline (Pima diabetes dataset)

In [ ]:
import pandas as pd
import numpy as np
import requests
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ["preg","glucose","bp","skin","insulin","bmi","pedigree","age","label"]
df = pd.read_csv(url, names=cols)
print(df.shape)
df.head()

In [ ]:
X = df.drop(columns="label"); y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
print("Test AUROC:", round(roc_auc_score(y_test, proba), 3))

cv = cross_val_score(model, X, y,
                     cv=StratifiedKFold(5, shuffle=True, random_state=42),
                     scoring="roc_auc")
print("5-fold CV AUROC:", cv.round(3), "| mean:", cv.mean().round(3))
print("Confusion matrix (test set):\n", confusion_matrix(y_test, model.predict(X_test)))

### Cell Health Score

In [ ]:
def cell_health_score(risk_proba: float):
    """Disease-risk probability (0-1) -> Cell Health Score (0-100) + category."""
    score = int(round(100 * (1 - risk_proba)))
    if score >= 80:    cat = "healthy"
    elif score >= 60:  cat = "stressed"
    elif score >= 40:  cat = "inflamed"
    else:              cat = "pre-diabetic-risk"
    return score, cat

sample = pd.DataFrame([[2, 121, 70, 25, 80, 28.5, 0.35, 29]], columns=X.columns)
p = model.predict_proba(sample)[0, 1]
score, cat = cell_health_score(p)
print(f"risk probability: {p:.2f}  ->  Cell Health Score: {score}/100 ({cat})")

In [ ]:
joblib.dump(model, "pima_model.joblib")
reloaded = joblib.load("pima_model.joblib")
print("saved + reloaded OK:", np.isclose(reloaded.predict_proba(sample)[0,1], p))

# Part B — real metabolomics data (MetaboLights MTBLS1: urine NMR, 132 people, T2D vs control)

In [ ]:
FTP = "https://ftp.ebi.ac.uk/pub/databases/metabolights/studies/public/MTBLS1/"

s1 = pd.read_csv(FTP + "s_MTBLS1.txt", sep="\t", low_memory=False)
labels = s1[["Sample Name", "Factor Value[Metabolic syndrome]"]].dropna()
labels.columns = ["sample", "group"]
labels["y"] = (labels["group"] == "diabetes mellitus").astype(int)
print(labels["group"].value_counts())

In [ ]:
m1 = pd.read_csv(FTP + "m_MTBLS1_metabolite_profiling_NMR_spectroscopy_v2_maf.tsv",
                 sep="\t", low_memory=False)
sample_cols = [c for c in m1.columns if c.startswith("ADG")]                                    
data = m1.set_index("chemical_shift")[sample_cols].T
data.index.name = "sample"

merged = data.merge(labels[["sample","y"]], left_index=True, right_on="sample").set_index("sample")
Xb = merged.drop(columns="y"); yb = merged["y"]
print("matrix:", Xb.shape, "| diabetic:", int(yb.sum()), "| control:", int((1-yb).sum()))

In [ ]:
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb, yb, test_size=0.25, stratify=yb, random_state=42)

model_b = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=0.1)),
])
model_b.fit(Xb_train, yb_train)

pb = model_b.predict_proba(Xb_test)[:, 1]
print("Test AUROC:", round(roc_auc_score(yb_test, pb), 3))
cvb = cross_val_score(model_b, Xb, yb,
                      cv=StratifiedKFold(5, shuffle=True, random_state=42),
                      scoring="roc_auc")
print("5-fold CV AUROC:", cvb.round(3), "| mean:", cvb.mean().round(3))
print("Confusion matrix:\n", confusion_matrix(yb_test, model_b.predict(Xb_test)))

joblib.dump(model_b, "mtbls1_metabolomics_model.joblib")
print("saved: mtbls1_metabolomics_model.joblib")

### Which signals drive the model

In [ ]:
coef = pd.Series(model_b.named_steps["clf"].coef_[0], index=Xb.columns)
top = coef.abs().sort_values(ascending=False).head(15)
print("Top 15 signals (chemical-shift bins, ppm):")
print(top)